# Pre-entrega 6 · Orquestador multi-agente

Esta demo ejecuta el Supervisor y los especialistas con OpenAI. La búsqueda consulta tres apuntes locales con referencias al NIST; los tiempos de respuesta son **datos sintéticos**. El notebook muestra delegaciones, herramientas, aportes y síntesis, no razonamiento interno privado.

Para reproducir: instalar `requirements-demo.txt`, configurar `.env` y ejecutar desde esta carpeta. Las celdas de modelo consumen API. Los outputs guardados permiten revisar el flujo sin volver a pagar llamadas.

In [1]:
import json
import logging
import platform
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

from IPython.display import Markdown, display

from config import Settings
from graph import run_analysis
from state import AnalysisRequest

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)
print("Fecha UTC:", datetime.now(timezone.utc).isoformat())
print("Python:", platform.python_version())
print("Modelo:", Settings().openai_model)
packages = {
    name: version(name)
    for name in ("langgraph", "langchain-openai", "openai", "pydantic")
}
print("Dependencias:", packages)

Fecha UTC: 2026-09-28T16:12:39.946557+00:00
Python: 3.12.13
Modelo: gpt-5.6-luna
Dependencias: {'langgraph': '1.2.11', 'langchain-openai': '1.6.2', 'openai': '3.19.2', 'pydantic': '2.13.5'}


## 1. Solicitud y datos

La pregunta necesita investigación conceptual y cálculo. Los valores van en un contrato validado, separados de la instrucción textual; el modelo no puede reemplazarlos al llamar la herramienta de cálculo.

In [2]:
request = AnalysisRequest.model_validate_json(
    Path("examples/comparison.json").read_text(encoding="utf-8")
)
print(request.model_dump_json(indent=2))

{
  "query": "Buscá cómo comparar dos grupos y analizá estos tiempos de respuesta. ¿Cuál tiene menor promedio y cuál es más consistente? Explicá media, mediana y dispersión con fuentes, sin inferir significancia estadística.",
  "groups": [
    {
      "name": "A",
      "unit": "ms",
      "values": [
        100.0,
        110.0,
        90.0,
        100.0,
        100.0
      ]
    },
    {
      "name": "B",
      "unit": "ms",
      "values": [
        80.0,
        120.0,
        100.0,
        90.0,
        110.0
      ]
    }
  ]
}


## 2. Ejecución real

El Supervisor decide el orden de las delegaciones. Cada especialista vuelve al Supervisor con un aporte propio. Solo se finaliza como completado cuando están los respaldos necesarios; hay límites de pasos para evitar ciclos indefinidos.

In [3]:
result = await run_analysis(request)
print("Completado:", result["completed"])
print("Duración (segundos):", result["seconds"])
assert result["completed"], result["response"]

INFO httpx2: HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


INFO graph: model agent=supervisor seconds=2.771


INFO httpx2: HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


INFO agents.common: model agent=research seconds=1.664


INFO agents.common: tool agent=research seconds=0.002


...

INFO graph: model agent=supervisor seconds=4.654


INFO httpx2: HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


INFO graph: model agent=synthesis seconds=3.659


INFO graph: total seconds=31.869


Completado: True
Duración (segundos): 31.868681199999628


## 3. Traza observable de delegación y herramientas

Estos eventos se generan durante la ejecución. No se escribe de antemano qué especialista participa primero.

In [4]:
for position, event in enumerate(result["events"], start=1):
    print(f"Paso {position}")
    print(json.dumps(event, ensure_ascii=False, indent=2))

Paso 1
{
  "agent": "supervisor",
  "action": "decision",
  "next_agent": "research",
  "instruction": "Buscá en el corpus local fuentes pertinentes y vigentes sobre cómo comparar dos grupos mediante media, mediana y desviación estándar muestral, y sobre la interpretación descriptiva de dispersión. Devolvé únicamente las fuentes verificadas con sus títulos, referencias y los puntos que respaldan; no calcules métricas ni infieras significancia estadística.",
  "rubric": "La respuesta debe aportar fuentes locales pertinentes y vigentes, verificadas, que permitan explicar media, mediana y dispersión sin afirmar pruebas de significancia.",
  "seconds": 2.7719997000003787
}
Paso 2
{
  "agent": "research",
  "action": "model",
  "seconds": 1.6646809000003486
}
Paso 3
{
  "agent": "research",
  "action": "tool",
  "tool": "buscar_fuentes",
  "arguments": {
    "query": "comparar dos grupos media mediana desviación estándar muestral interpretación descriptiva dispersión"
  },
  "result": {
   

## 4. Estado compartido y procedencia

Los aportes conservan el agente que los produjo, su estado y los resultados de las herramientas. Las revisiones no borran la procedencia anterior.

In [5]:
print(json.dumps(result["contributions"], ensure_ascii=False, indent=2))

[
  {
    "agent": "research",
    "status": "complete",
    "narrative": "- **Título:** *Dispersión, desviación estándar y consistencia*  \n  **Referencia:** NIST/ITL, *Engineering Statistics Handbook*, sección EDA sobre dispersión. Fuente local: https://www.itl.nist.gov/div898/handbook/eda/section3/eda356.htm  \n  **Puntos respaldados:**\n  - La dispersión describe cuánto varían las observaciones.\n  - La desviación estándar muestral se obtiene como la raíz cuadrada de la varianza muestral; conserva las unidades originales y requiere al menos dos observaciones.\n  - Para comparar la consistencia de los grupos A y B, con la misma variable y unidad, una menor desviación estándar descriptivamente indica menor dispersión.\n  - Una menor dispersión no implica un menor promedio.\n  - Los valores extremos pueden influir considerablemente en la desviación estándar.\n\n- **Título:** *Promedio y mediana: medidas de centro*  \n  **Referencia:** NIST/ITL, *Engineering Statistics Handbook*, secci

## 5. Respuesta final

Las tablas numéricas y referencias proceden de las herramientas. La comparación es descriptiva: no prueba significancia estadística ni anticipa el rendimiento futuro.

In [6]:
display(Markdown(result["response"]))

Descriptivamente, ambos grupos tienen el mismo promedio y la misma mediana, por lo que ninguno presenta un promedio menor. La mediana representa el valor central de los datos ordenados y es menos sensible a valores extremos que la media. La dispersión indica cuánto varían las observaciones; la desviación estándar muestral la expresa en las mismas unidades que los tiempos. A tiene menor desviación estándar que B, por lo que es el grupo más consistente según este criterio. Esta comparación no evalúa significancia estadística, causalidad ni desempeño futuro. Fuentes: “Promedio y mediana: medidas de centro” (NIST: https://www.itl.nist.gov/div898/handbook/eda/section3/eda351.htm), “Dispersión, desviación estándar y consistencia” (NIST: https://www.itl.nist.gov/div898/handbook/eda/section3/eda356.htm) y “Comparación descriptiva y límites de la inferencia” (NIST: https://www.itl.nist.gov/div898/handbook/eda/section3/eda353.htm).

| Grupo | Unidad | n | Promedio | Mediana | Desviación muestral |
| --- | --- | ---: | ---: | ---: | ---: |
| A | ms | 5 | 100 | 100 | 7.07107 |
| B | ms | 5 | 100 | 100 | 15.8114 |

Menor promedio: empate. Menor dispersión: A.
Comparación descriptiva; no demuestra significancia estadística ni causalidad.

Fuentes:
- [Dispersión, desviación estándar y consistencia](https://www.itl.nist.gov/div898/handbook/eda/section3/eda356.htm) (dispersion)
- [Promedio y mediana: medidas de centro](https://www.itl.nist.gov/div898/handbook/eda/section3/eda351.htm) (centro)
- [Comparación descriptiva y límites de la inferencia](https://www.itl.nist.gov/div898/handbook/eda/section3/eda353.htm) (comparacion)

## 6. Verificación independiente y evidencia

Para este dataset conocido, los dos promedios y las dos medianas son 100 ms. Los desvíos muestrales esperados son aproximadamente 7,071 ms y 15,811 ms. La menor dispersión corresponde a A.

Se conserva el resultado de esta pasada en JSON. Antes de compartir una demo con datos propios, revisar que la traza no contenga información sensible.

In [7]:
from math import isclose
from statistics import mean, median, stdev

expected = [
    {
        "group": group.name,
        "mean": mean(group.values),
        "median": median(group.values),
        "stdev": stdev(group.values),
    }
    for group in request.groups
]
assert expected[0]["mean"] == expected[1]["mean"] == 100
assert expected[0]["stdev"] < expected[1]["stdev"]
latest = {item["agent"]: item for item in result["contributions"]}
assert latest["research"]["status"] == latest["analysis"]["status"] == "complete"
assert latest["research"]["sources"]
actual = latest["analysis"]["stats"]
assert len(actual) == len(expected) == 2
for measured, reference in zip(actual, expected, strict=True):
    assert measured["name"] == reference["group"]
    for metric in ("mean", "median", "stdev"):
        assert isclose(measured[metric], reference[metric], rel_tol=1e-9, abs_tol=1e-9)
used_tools = {event["tool"] for event in result["events"] if event["action"] == "tool"}
assert {"buscar_fuentes", "calcular_estadisticas"} <= used_tools
print(json.dumps(expected, ensure_ascii=False, indent=2))

evidence = Path("evidencias")
evidence.mkdir(exist_ok=True)
trace_path = evidence / "demo_result.json"
result["run_metadata"] = {
    "utc": datetime.now(timezone.utc).isoformat(),
    "model": Settings().openai_model,
    "python": platform.python_version(),
    "packages": packages,
    "synthetic_data": True,
}
trace_path.write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Traza:", trace_path.as_posix())

[
  {
    "group": "A",
    "mean": 100.0,
    "median": 100.0,
    "stdev": 7.0710678118654755
  },
  {
    "group": "B",
    "mean": 100.0,
    "median": 100.0,
    "stdev": 15.811388300841896
  }
]
Traza: evidencias/demo_result.json


## Alcance

Esta ejecución demuestra una comparación de dos muestras y la participación de especialistas. Los tests locales cubren adicionalmente validaciones, refinamientos, errores y límites con un modelo simulado. Una pasada real no demuestra una tasa de éxito general. La base documental es pequeña, la búsqueda es local y no hay autenticación, servicio web ni memoria entre ejecuciones.